In [1]:
#Load Libraries
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from scipy.interpolate import interp1d
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
N_POINTS     = 100       # points per surface (upper + lower) → input dim = 200
LATENT_DIM   = 32        # VAE bottleneck dimension
EPOCHS       = 150       # training epochs
BATCH_SIZE   = 128
LR           = 1e-3
BETA         = 1.0       # KL weight in ELBO loss (beta-VAE; 1.0 = standard VAE)
N_GENERATE   = 16        # novel airfoils to generate
SEED         = 42

torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
 
# colour palette
C1, C2, C3, C4, C5 = "#2196F3", "#F44336", "#4CAF50", "#FF9800", "#9C27B0"
DARK   = "#0d1117"
PANEL  = "#161b22"
BORDER = "#30363d"

In [3]:
shapes  = np.load("curated_airfoils.npz")["shapes"]   # (19164, 1001, 2)
classes = np.load("curated_airfoils.npz")["classes"]  # (19164,)
print(f"  Loaded {len(shapes):,} airfoils")

  Loaded 19,164 airfoils


In [4]:
# Unified cosine-spaced x grid from 0 → 1
x_grid = (1 - np.cos(np.linspace(0, np.pi, N_POINTS))) / 2   # (N_POINTS,)

In [5]:
def airfoil_to_vector(shape: np.ndarray):
    """
    Convert a (1001, 2) raw airfoil array into a (2*N_POINTS,) flat vector:
      [y_upper(x_grid), y_lower(x_grid)]
 
    The G2Aero dataset stores points starting at the trailing edge (x~1),
    going around the upper surface to the leading edge (x~0), then back
    along the lower surface to the trailing edge.  The leading-edge index
    is therefore the point with the minimum x value.
 
    Returns None if interpolation fails or the shape is degenerate.
    """
    x, y   = shape[:, 0], shape[:, 1]
    le_idx = int(np.argmin(x))
 
    # G2Aero layout: indices 0..le_idx are the LOWER surface (TE to LE),
    # and indices le_idx..end are the UPPER surface (LE to TE).
    # Reverse the first half so both surfaces run LE to TE.
    x_lo = x[:le_idx + 1][::-1].copy()   # lower: TE->LE reversed = LE->TE
    y_lo = y[:le_idx + 1][::-1].copy()
 
    x_up = x[le_idx:].copy()             # upper: already LE->TE
    y_up = y[le_idx:].copy()
 
    # Remove duplicate x values while preserving point order.
    # np.unique returns sorted first-occurrence indices, which keeps
    # the original ordering for monotone arrays.
    def dedup(xarr, yarr):
        _, keep = np.unique(xarr, return_index=True)
        return xarr[keep], yarr[keep]
 
    x_up, y_up = dedup(x_up, y_up)
    x_lo, y_lo = dedup(x_lo, y_lo)
 
    # Need at least 3 points and strictly increasing x on each surface
    if len(x_up) < 3 or len(x_lo) < 3:
        return None
    if not (np.all(np.diff(x_up) > 0) and np.all(np.diff(x_lo) > 0)):
        return None
 
    # Interpolate onto the shared cosine x-grid
    try:
        yu = np.interp(x_grid, x_up, y_up)
        yl = np.interp(x_grid, x_lo, y_lo)
    except Exception:
        return None
 
    vec = np.concatenate([yu, yl]).astype(np.float32)
 
    # Physical sanity checks
    thickness = yu - yl
 
    # No self-intersection (allow small numerical tolerance)
    if np.any(thickness < -0.005):
        return None
 
    # Realistic thickness: 0.5% to 60% chord
    max_t = float(np.max(thickness))
    if max_t < 0.005 or max_t > 0.60:
        return None
 
    # No surface point more than 40% chord off the chord line
    if np.any(np.abs(vec) > 0.40):
        return None
 
    return vec

def vector_to_surfaces(vec: np.ndarray):
    """Reverse: split (2*N_POINTS,) vector back into upper/lower y arrays."""
    yu = vec[:N_POINTS]
    yl = vec[N_POINTS:]
    return yu, yl

In [6]:
print("  Converting airfoils to flat y-vectors ...")
vectors, valid_idx, valid_cls = [], [], []
for i, shape in enumerate(shapes):
    v = airfoil_to_vector(shape)
    if v is not None:
        vectors.append(v)
        valid_idx.append(i)
        valid_cls.append(classes[i])
X_raw     = np.array(vectors) if vectors else np.empty((0, 2 * N_POINTS), dtype=np.float32)
valid_cls = np.array(valid_cls)
print(f"  Valid airfoils after QC : {len(X_raw):,}")

  Converting airfoils to flat y-vectors ...
  Valid airfoils after QC : 19,164


In [7]:
if len(X_raw) == 0:
    raise RuntimeError(
        "No airfoils survived the QC filter. "
    )

In [8]:
# Normalise to zero mean / unit std (per-feature, across dataset)
scaler  = StandardScaler()
X_norm  = scaler.fit_transform(X_raw)

In [9]:
# Geometric labels for colouring latent space
max_t_arr  = np.array([v[:N_POINTS].max() - v[N_POINTS:].min()         # approx thickness
                        for v in X_raw])
max_c_arr  = np.array([((v[:N_POINTS] + v[N_POINTS:]) / 2).max()        # approx max camber
                        for v in X_raw])
le_t_arr   = np.array([v[2] - v[N_POINTS + 2]                            # LE thickness proxy
                        for v in X_raw])

In [10]:
n_total = len(X_norm)
n_train = int(0.9 * n_total)
idx     = rng.permutation(n_total)
tr, va  = idx[:n_train], idx[n_train:]
 
X_train = torch.tensor(X_norm[tr], dtype=torch.float32)
X_val   = torch.tensor(X_norm[va], dtype=torch.float32)
 
train_loader = DataLoader(TensorDataset(X_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val),   batch_size=BATCH_SIZE, shuffle=False)

In [11]:
INPUT_DIM = 2 * N_POINTS 

class ResBlock1D(nn.Module):
    """1-D residual block: Conv → BN → ReLU → Conv → BN + skip."""
    def __init__(self, channels: int, kernel: int = 3):
        super().__init__()
        pad = kernel // 2
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel, padding=pad),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Conv1d(channels, channels, kernel, padding=pad),
            nn.BatchNorm1d(channels),
        )
        self.relu = nn.ReLU()
 
    def forward(self, x):
        return self.relu(x + self.block(x)) #Use the RELU function for forward processing

In [12]:
class AirfoilEncoder(nn.Module):
    """
    Conv1D encoder.
    Input  : (B, 1, 200)
    Output : mu (B, LATENT_DIM), log_var (B, LATENT_DIM)
    """
    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            # (B,  1, 200) → (B, 32, 200)
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(),
            ResBlock1D(32),
            # (B, 32, 200) → (B, 64, 100)
            nn.Conv1d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(),
            ResBlock1D(64),
            # (B, 64, 100) → (B, 128, 50)
            nn.Conv1d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            ResBlock1D(128),
            # (B, 128, 50) → (B, 256, 25)
            nn.Conv1d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.flat_dim = 256 * 25
        self.fc_mu     = nn.Linear(self.flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_dim, latent_dim)
 
    def forward(self, x):                        # x: (B, 1, 200)
        h  = self.conv(x).flatten(1)             # (B, 6400)
        return self.fc_mu(h), self.fc_logvar(h)  # (B, LD), (B, LD)

In [16]:
class AirfoilDecoder(nn.Module):
    """
    ConvTranspose1D decoder.
    Input  : z (B, LATENT_DIM)
    Output : (B, 1, 200)
    """
    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.flat_dim = 256 * 25
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, self.flat_dim),
            nn.ReLU(),
        )
        self.deconv = nn.Sequential(
            # (B, 256, 25) → (B, 128, 50)
            nn.ConvTranspose1d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            ResBlock1D(128),
            # (B, 128, 50) → (B, 64, 100)
            nn.ConvTranspose1d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(),
            ResBlock1D(64),
            # (B, 64, 100) → (B, 32, 200)
            nn.ConvTranspose1d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(),
            ResBlock1D(32),
            # (B, 32, 200) → (B, 1, 200)
            nn.Conv1d(32, 1, kernel_size=5, padding=2),
        )
 
    def forward(self, z):
        h = self.fc(z).view(-1, 256, 25)
        return self.deconv(h)                    # (B, 1, 200)

In [13]:
class AirfoilVAE(nn.Module):
    """Full VAE with reparametrisation trick."""
    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.encoder = AirfoilEncoder(latent_dim)
        self.decoder = AirfoilDecoder(latent_dim)
        self.latent_dim = latent_dim
 
    def reparametrize(self, mu, log_var):
        """z = mu + eps * std,  eps ~ N(0,I)."""
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std
 
    def forward(self, x):
        mu, log_var = self.encoder(x)
        z           = self.reparametrize(mu, log_var)
        x_hat       = self.decoder(z)
        return x_hat, mu, log_var
 
    def encode(self, x):
        mu, log_var = self.encoder(x)
        return self.reparametrize(mu, log_var), mu, log_var
 
    def decode(self, z):
        return self.decoder(z)
 
    def sample(self, n: int, device="cpu"):
        """Sample n novel airfoils from the prior N(0, I)."""
        z = torch.randn(n, self.latent_dim, device=device)
        return self.decoder(z)

In [14]:
def vae_loss(x, x_hat, mu, log_var, beta: float = BETA):
    """
    ELBO = Reconstruction loss + beta * KL divergence
    Reconstruction: mean squared error over all y-coordinates
    KL: closed-form  -0.5 * sum(1 + log_var - mu^2 - exp(log_var))
    """
    recon = nn.functional.mse_loss(x_hat, x, reduction="sum") / x.shape[0]
    kl    = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.shape[0]
    return recon + beta * kl, recon, kl

In [17]:
model  = AirfoilVAE(LATENT_DIM)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n  VAE architecture :")
print(f"    Input dim   : {INPUT_DIM}")
print(f"    Latent dim  : {LATENT_DIM}")
print(f"    Parameters  : {n_params:,}")
print(f"    Beta (KL)   : {BETA}")


  VAE architecture :
    Input dim   : 200
    Latent dim  : 32
    Parameters  : 1,228,097
    Beta (KL)   : 1.0


Train the Data

In [18]:
device    = "cpu"
model     = model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

In [19]:
# Cosine annealing with warm restarts
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=2, eta_min=1e-5)

In [20]:
# KL annealing — ramp beta from 0 → BETA over first 30 epochs
# Prevents posterior collapse at the start of training
def beta_schedule(epoch: int, warmup: int = 30):
    return min(BETA, BETA * epoch / warmup)

In [21]:
history = {"train_loss": [], "val_loss": [],
           "train_recon": [], "train_kl": []}
 
best_val_loss = float("inf")
best_state    = None

In [25]:
for epoch in range(1, EPOCHS + 1):
    beta_e = beta_schedule(epoch)
 
    # ── train ──
    model.train()
    t_loss, t_recon, t_kl = 0.0, 0.0, 0.0
    for (xb,) in train_loader:
        xb = xb.unsqueeze(1).to(device)       # (B, 1, 200)
        optimizer.zero_grad()
        x_hat, mu, lv = model(xb)
        loss, recon, kl = vae_loss(xb, x_hat, mu, lv, beta=beta_e)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        t_loss  += loss.item()
        t_recon += recon.item()
        t_kl    += kl.item()
    scheduler.step()
 
    # ── validate ──
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for (xb,) in val_loader:
            xb = xb.unsqueeze(1).to(device)
            x_hat, mu, lv = model(xb)
            loss, _, _ = vae_loss(xb, x_hat, mu, lv, beta=beta_e)
            v_loss += loss.item()
 
    t_loss  /= len(train_loader)
    t_recon /= len(train_loader)
    t_kl    /= len(train_loader)
    v_loss  /= len(val_loader)
 
    history["train_loss"].append(t_loss)
    history["val_loss"].append(v_loss)
    history["train_recon"].append(t_recon)
    history["train_kl"].append(t_kl)
 
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
 
    if epoch % 25 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"  Epoch {epoch:>4}/{EPOCHS}  "
              f"train={t_loss:.4f}  val={v_loss:.4f}  "
              f"recon={t_recon:.4f}  kl={t_kl:.4f}  "
              f"beta={beta_e:.3f}  lr={lr_now:.2e}")
# Restore best weights
model.load_state_dict(best_state)
model.eval()
print(f"\n  Best val loss : {best_val_loss:.4f}")

  Epoch    1/150  train=2.0615  val=1.5606  recon=1.7010  kl=10.8148  beta=0.033  lr=2.08e-05
  Epoch   25/150  train=9.0923  val=9.3062  recon=3.7335  kl=6.4306  beta=0.833  lr=9.34e-04
  Epoch   50/150  train=9.5984  val=9.7751  recon=3.8773  kl=5.7211  beta=1.000  lr=6.94e-04
  Epoch   75/150  train=9.2568  val=9.1962  recon=3.5954  kl=5.6614  beta=1.000  lr=3.77e-04
  Epoch  100/150  train=8.9893  val=8.8828  recon=3.4020  kl=5.5873  beta=1.000  lr=1.12e-04
  Epoch  125/150  train=8.8916  val=8.9286  recon=3.3255  kl=5.5661  beta=1.000  lr=1.00e-03
  Epoch  150/150  train=9.4926  val=9.4474  recon=3.8499  kl=5.6428  beta=1.000  lr=9.74e-04

  Best val loss : 1.5606


In [26]:
all_z, all_mu = [], []
X_all_t = torch.tensor(X_norm, dtype=torch.float32)
 
with torch.no_grad():
    for start in range(0, len(X_all_t), 512):
        xb = X_all_t[start:start + 512].unsqueeze(1).to(device)
        z, mu, _ = model.encode(xb)
        all_z.append(z.cpu().numpy())
        all_mu.append(mu.cpu().numpy())
        
Z_all  = np.concatenate(all_z,  axis=0)   # (N_valid, LATENT_DIM)
MU_all = np.concatenate(all_mu, axis=0)   # (N_valid, LATENT_DIM)
 
# Reconstruction quality on validation set
with torch.no_grad():
    X_val_np    = X_norm[va]
    X_val_t     = torch.tensor(X_val_np, dtype=torch.float32).unsqueeze(1).to(device)
    X_recon_t, mu_val, lv_val = model(X_val_t)
    X_recon_np  = X_recon_t.squeeze(1).cpu().numpy()
    

In [27]:
# Inverse-normalise for geometry
X_val_orig   = scaler.inverse_transform(X_val_np)
X_recon_orig = scaler.inverse_transform(X_recon_np)
 
recon_mse  = np.mean((X_val_orig - X_recon_orig) ** 2)
recon_rmse = np.sqrt(recon_mse)
print(f"  Reconstruction RMSE (val, y-coords) : {recon_rmse:.6f}")

  Reconstruction RMSE (val, y-coords) : 0.003194


In [28]:
# PCA of latent space for 2-D visualisation
pca_lat = PCA(n_components=2, random_state=SEED)
Z_pca   = pca_lat.fit_transform(Z_all)
print(f"  Latent PCA explained variance : {pca_lat.explained_variance_ratio_.sum()*100:.1f}%")

  Latent PCA explained variance : 13.2%
